In [ ]:
import pandas as pd

df = pd.read_csv('data/accepted_2007_to_2018Q4.csv.gz', compression='gzip', low_memory=False)

print(df.shape)
print(df.head(3))

In [ ]:
print(df.dtypes.value_counts())
print("\n")
print(df.isnull().sum().sort_values(ascending=False).head(20))

In [ ]:
# Drop columns where more than 50% of data is missing
threshold = len(df) * 0.5
df_clean = df.dropna(axis=1, thresh=threshold)

print("Before:", df.shape)
print("After:", df_clean.shape)

In [ ]:
cols_needed = [
    'loan_amnt', 'funded_amnt', 'term', 'int_rate', 'grade', 'sub_grade',
    'emp_length', 'home_ownership', 'annual_inc', 'loan_status',
    'dti', 'delinq_2yrs', 'purpose', 'revol_util', 'total_acc',
    'application_type', 'mort_acc', 'pub_rec_bankruptcies'
]

df_clean = df_clean[cols_needed].copy()

df_sample = df_clean.sample(n=50000, random_state=42)

print(df_sample.shape)
print(df_sample.head(3))

In [ ]:
print(df_sample.dtypes)
print("\n")
print(df_sample.isnull().sum())

In [ ]:
# Drop rows where nulls are very small
df_sample = df_sample.dropna(subset=['loan_amnt','funded_amnt','term','int_rate',
                                      'grade','sub_grade','home_ownership',
                                      'annual_inc','loan_status','delinq_2yrs',
                                      'purpose','total_acc','application_type'])

# Fill emp_length with Unknown
df_sample['emp_length'] = df_sample['emp_length'].fillna('Unknown')

# Fill mort_acc and pub_rec_bankruptcies with 0
df_sample['mort_acc'] = df_sample['mort_acc'].fillna(0)
df_sample['pub_rec_bankruptcies'] = df_sample['pub_rec_bankruptcies'].fillna(0)

# Fill dti and revol_util with median
df_sample['dti'] = df_sample['dti'].fillna(df_sample['dti'].median())
df_sample['revol_util'] = df_sample['revol_util'].fillna(df_sample['revol_util'].median())

print(df_sample.isnull().sum())
print("\nShape:", df_sample.shape)

In [ ]:
print(df_sample['loan_status'].value_counts())

In [ ]:
default_statuses = [
    'Charged Off',
    'Late (31-120 days)',
    'Does not meet the credit policy. Status:Charged Off'
]

df_sample['default_flag'] = df_sample['loan_status'].apply(
    lambda x: 1 if x in default_statuses else 0
)

print(df_sample['default_flag'].value_counts())
print("\nDefault rate:", round(df_sample['default_flag'].mean() * 100, 2), "%")

In [ ]:
# Loan to income ratio
df_sample['loan_to_income'] = df_sample['loan_amnt'] / df_sample['annual_inc']

# DTI bucket
df_sample['dti_bucket'] = pd.cut(
    df_sample['dti'],
    bins=[0, 10, 20, 35, 999],
    labels=['Low', 'Medium', 'High', 'Very High']
)

print(df_sample[['loan_to_income', 'dti_bucket']].describe())
print("\n")
print(df_sample['dti_bucket'].value_counts())

In [ ]:
# Check how many zero income rows exist
print("Zero income rows:", (df_sample['annual_inc'] == 0).sum())

# Drop them
df_sample = df_sample[df_sample['annual_inc'] > 0]

# Recalculate loan_to_income
df_sample['loan_to_income'] = df_sample['loan_amnt'] / df_sample['annual_inc']

print("\nShape after fix:", df_sample.shape)
print("\nloan_to_income stats:")
print(df_sample['loan_to_income'].describe())

In [ ]:
# Check how many rows are above ratio of 10
print("Rows with loan_to_income > 10:", (df_sample['loan_to_income'] > 10).sum())

# Cap at 10 — no real lender approves beyond this
df_sample = df_sample[df_sample['loan_to_income'] <= 10]

print("\nShape after cap:", df_sample.shape)
print("\nloan_to_income stats:")
print(df_sample['loan_to_income'].describe())

In [ ]:
df_sample.to_csv('data/loans_clean.csv', index=False)

print("File saved.")
print("Final shape:", df_sample.shape)
print("Default rate:", round(df_sample['default_flag'].mean() * 100, 2), "%")